In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Chandni_Chowk_Delhi_IITM_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,217.0,109.0,170.0,83.0,NaN,106.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2,327.0,136.0,176.0,124.0,NaN,156.0,NaN,NaN,NaN,NaN,NaN,NaN
2,3,304.0,NaN,144.0,162.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,296.0,214.0,123.0,115.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,321.0,NaN,132.0,148.0,199.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,374.0,229.0,119.0,144.0,224.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,370.0,220.0,NaN,131.0,178.0,138.0,NaN,NaN,NaN,NaN,NaN,NaN
7,8,287.0,128.0,205.0,159.0,140.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,316.0,NaN,103.0,219.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,NaN,NaN,171.0,NaN,202.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape


(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))


In [6]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())



In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()


,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,198.526316,109.000000,170.0,144.769231,153.785714,73.333333,15.5,15.5,15.0,15.5,15.0,15.5
1,2,198.526316,136.000000,176.0,124.000000,153.785714,73.333333,15.5,15.5,15.0,15.5,15.0,15.5
2,3,198.526316,166.238095,144.0,162.000000,153.785714,73.333333,15.5,15.5,15.0,15.5,15.0,15.5
3,4,198.526316,214.000000,123.0,144.769231,153.785714,73.333333,15.5,15.5,15.0,15.5,15.0,15.5
4,5,198.526316,166.238095,132.0,148.000000,199.000000,73.333333,15.5,15.5,15.0,15.5,15.0,15.5
